# Poly-3 baseline at fixed $n=500$, all DGPs

Runs \textsc{Pds}-\textsc{Lasso} (poly-3) at $n=500$ for every DGP at the full
replication count, so the fixed-$n$ tables can carry a complete 500-rep poly-3
row in place of the 25-rep placeholder. Poly-2 already exists at 500 reps
(`pds_poly2_fixedn.csv`) and is left untouched.

Poly-3 is heavy: the degree-3 dictionary is ~23,400 columns, so each
replication is two wide LASSO fits. The run is **resumable** and saves after
every DGP, so an interrupt loses at most one design. Writes
`pds_poly3_fixedn.csv` (labelled, same schema as the poly-2 file).


## Setup


In [ ]:
import time
import numpy as np, pandas as pd
import config as C
from dgp import DGP_REGISTRY
from evaluate import evaluate
from srpds_poly import pds_lasso_poly3

N      = C.N_HEADLINE      # 500
N_REPS = C.N_REPS_MAIN     # 500
RD     = C.RESULTS_DIR
print('poly-3 only | n =', N, '| n_rep =', N_REPS, '| results dir:', RD.resolve())


## Run (per-DGP, resumable)

Each replication draws a fresh dataset at its seed and runs poly-3; we keep
$\hat\beta_0$ and its interval, then reduce with the shared `evaluate`. Reps
that fail (rare, from a near-singular design at this width) are dropped and
counted.


In [ ]:
REP_PKL = RD / 'poly3_fixedn_reps.pkl'      # raw per-rep store (resume cache)
if REP_PKL.exists():
    reps = pd.read_pickle(REP_PKL)
    done = set(reps['dgp'].unique())
    print('resuming; done:', sorted(done))
else:
    reps, done = pd.DataFrame(), set()

t0 = time.time()
for dgp_key, e in DGP_REGISTRY.items():
    if dgp_key in done:
        continue
    rows, n_fail = [], 0
    for r in range(N_REPS):
        X, d, y, _ = e['fn'](n=N, p=C.P, s=C.S, beta0=C.BETA0, seed=r)
        try:
            res = pds_lasso_poly3(X, d, y, N, C.P)
            rows.append({'dgp': dgp_key, 'seed': r, 'beta_hat': res['beta_hat'],
                         'ci_low': res['ci_low'], 'ci_high': res['ci_high'],
                         'beta0': C.BETA0, 'failed': False})
        except Exception:
            n_fail += 1
            rows.append({'dgp': dgp_key, 'seed': r, 'beta_hat': np.nan,
                         'ci_low': np.nan, 'ci_high': np.nan,
                         'beta0': C.BETA0, 'failed': True})
    reps = pd.concat([reps, pd.DataFrame(rows)], ignore_index=True)
    reps.to_pickle(REP_PKL)
    print(f'  {dgp_key}: {N_REPS-n_fail}/{N_REPS} ok  '
          f'[{(time.time()-t0)/60:.1f} min]', flush=True)
print('\npoly-3 fixed-n reps complete')


## Reduce to bias / RMSE / coverage and write the labelled CSV

`evaluate` is the same reducer used for every other method, so these rows drop
straight into the fixed-$n$ tables. Labels match the table row exactly.


In [ ]:
reps = pd.read_pickle(REP_PKL)
out = []
for dgp_key, g in reps.groupby('dgp'):
    ok = g[~g['failed']]
    m = evaluate(ok, C.BETA0)
    m.update({'dgp': dgp_key, 'dgp_label': DGP_REGISTRY[dgp_key]['label'],
              'estimator': 'pds_poly3', 'est_label': 'PDS-LASSO (poly-3)',
              'n': N, 'n_rep': int(len(ok))})
    out.append(m)
poly3 = pd.DataFrame(out)
poly3.to_csv(RD / 'pds_poly3_fixedn.csv', index=False)
print('wrote pds_poly3_fixedn.csv  (', len(poly3), 'rows )\n')

# view in paper order (mild->severe): weak=dgp8, mild=dgp6, severe=dgp7
order = ['dgp1','dgp2','dgp3','dgp4','dgp5','dgp8','dgp6','dgp7']
view = (poly3.assign(o=poly3['dgp'].map({k:i for i,k in enumerate(order)}))
             .sort_values('o'))
print(view[['dgp','dgp_label','bias','rmse','coverage','n_rep']].to_string(index=False))


## Done

`pds_poly3_fixedn.csv` now holds the full 500-rep poly-3 row for all eight
designs. In Tables 3--5 replace the daggered placeholder cells with these
values and remove the `$^{\dagger}$` markers and the
``degree-3 baseline at $n_{\text{rep}}=25$'' clause from the three captions.
